본 프로젝트는 LINCS L1000 데이터에서 ctl_vector를 diseased-like state,
trt_cp를 compound-treated state로 간주한다.

모델은 diseased-like signature와 drug treatment 조건이 주어졌을 때
treated perturbation signature를 예측하도록 학습한다.

테스트 단계에서는 diseased signature와 target treated signature가 주어졌을 때,
후보 drug별 predicted treated signature를 생성하고,
target treated signature와의 cosine similarity를 기반으로 drug ranking을 수행한다.

## Config

In [1]:
import pandas as pd

## Load Data

In [6]:
x = pd.read_csv("./processed/X.csv")
metadata = pd.read_csv("./processed/metadata_X.csv")
y = pd.read_parquet("./processed/y.parquet")

In [7]:
x.head()

,Unnamed: 0,pert_id,cell_iname,pert_dose_num,pert_time_num
0,ABY001_A549_XH:BRD-A61304759:0.625:24,BRD-A61304759,A549,0.625,24.0
1,ABY001_A549_XH:BRD-A61304759:10:24,BRD-A61304759,A549,10.000,24.0
2,ABY001_A549_XH:BRD-A61304759:2.5:24,BRD-A61304759,A549,2.500,24.0
3,ABY001_A549_XH:BRD-A90490067:0.625:24,BRD-A90490067,A549,0.625,24.0
4,ABY001_A549_XH:BRD-A90490067:10:24,BRD-A90490067,A549,10.000,24.0


In [8]:
metadata.head()

,Unnamed: 0,bead_batch,nearest_dose,pert_dose,pert_dose_unit,pert_idose,pert_itime,pert_time,pert_time_unit,cell_mfc_name,...,det_plates,distil_ids,build_name,project_code,cmap_name,is_exemplar_sig,is_ncs_sig,is_null_sig,pert_time_num,pert_dose_num
0,ABY001_A549_XH:BRD-A61304759:0.625:24,b15,0.66,0.625,uM,0.66 uM,24 h,24.0,h,A549,...,ABY001_A549_XH_X1_B15,ABY001_A549_XH_X1_B15:H15|ABY001_A549_XH_X1_B1...,NaN,ABY,tanespimycin,0,1.0,0.0,24.0,0.625
1,ABY001_A549_XH:BRD-A61304759:10:24,b15,10.00,10.000,uM,10 uM,24 h,24.0,h,A549,...,ABY001_A549_XH_X1_B15,ABY001_A549_XH_X1_B15:H13|ABY001_A549_XH_X1_B1...,NaN,ABY,tanespimycin,0,1.0,0.0,24.0,10.000
2,ABY001_A549_XH:BRD-A61304759:2.5:24,b15,2.50,2.500,uM,2.5 uM,24 h,24.0,h,A549,...,ABY001_A549_XH_X1_B15,ABY001_A549_XH_X1_B15:H14|ABY001_A549_XH_X1_B1...,NaN,ABY,tanespimycin,0,1.0,0.0,24.0,2.500
3,ABY001_A549_XH:BRD-A90490067:0.625:24,b15,0.66,0.625,uM,0.66 uM,24 h,24.0,h,A549,...,ABY001_A549_XH_X1_B15,ABY001_A549_XH_X1_B15:F15|ABY001_A549_XH_X1_B1...,NaN,ABY,fulvestrant,0,1.0,0.0,24.0,0.625
4,ABY001_A549_XH:BRD-A90490067:10:24,b15,10.00,10.000,uM,10 uM,24 h,24.0,h,A549,...,ABY001_A549_XH_X1_B15,ABY001_A549_XH_X1_B15:D15|ABY001_A549_XH_X1_B1...,NaN,ABY,fulvestrant,0,1.0,0.0,24.0,10.000


In [9]:
y.head()

rid,10007,1001,10013,10038,10046,10049,10051,10057,10058,10059,...,9918,9924,9926,9928,993,994,9943,9961,998,9988
ABY001_A549_XH:BRD-A61304759:0.625:24,0.267136,-0.456491,2.165093,0.229504,-0.124224,1.576591,-1.828272,-0.225401,1.707536,-0.433627,...,1.564901,0.421255,-2.748089,-2.485454,-3.207149,-5.228687,-0.181523,-1.382612,-1.131088,1.328648
ABY001_A549_XH:BRD-A61304759:10:24,-0.117392,0.200503,3.915682,0.172298,-0.620917,1.896570,-0.351663,2.388200,2.241555,0.882148,...,2.835503,-2.048177,-1.072422,-1.558214,-3.306095,-1.125444,-0.274837,-2.573916,-1.041967,0.582418
ABY001_A549_XH:BRD-A61304759:2.5:24,0.703145,-0.459059,2.882389,-0.590942,-0.809936,1.652521,-1.464148,1.901619,1.833337,-0.166399,...,2.853060,-1.136871,-1.968534,-1.382610,-3.071453,-2.076343,-0.108275,-3.197631,0.172597,0.360270
ABY001_A549_XH:BRD-A90490067:0.625:24,0.911685,-0.032964,-0.585127,-0.394690,-0.081405,0.156198,1.680390,0.157106,-1.150557,-0.833759,...,-1.232962,-1.763807,0.304337,0.471825,-0.021767,-0.316735,0.263729,-0.352919,0.556488,0.320331
ABY001_A549_XH:BRD-A90490067:10:24,1.412661,0.352456,0.386300,-1.181262,3.863513,0.755301,-1.086625,-0.230203,-1.392281,-3.727853,...,-0.482366,-0.636021,-1.075500,-1.617027,-1.222583,-0.952263,-1.018351,0.424994,-0.618140,1.483994


## Filter data

## Build features

## Split

## Train

## Evaluate

## Drug ranking